# SoilSight · 02b · Uncertainty v2: which signal predicts the error?

The first "review needed" rule used how much a sample's photos disagree. On the calibration samples it flagged 96%
and the accepted ones were *worse*. This notebook compares five difficulty signals against real out-of-fold errors,
keeps the one that tracks error best, calibrates its range, and sets the review threshold so a chosen share of
samples is auto-accepted. CPU only (Accelerator **None**), Internet **On**, `HF_TOKEN` attached. Takes about a minute.
The test split stays untouched.

In [ ]:
# ---- settings ----
CACHE_REPO_NAME = "photogranulometry-cache"
MODEL_REPO_NAME = "soilsight-models"
K               = 10      # neighbours used by the kNN signals
COVERAGE        = 0.90
TARGET_ACCEPT   = 0.50    # share of samples to auto-accept (the most confident ones)


In [ ]:
# ---- unpack ----
import base64, io, tarfile, sys, os, json
PAYLOAD = "H4sIAA87/WoC/+1c/3LbRpLO33yKWabuAjgUTFK2bGuXqciJ4qTOsVW2N7GXxUKBBEgiJAEYACnJOt073Svck93X3TPAgKTkZNebyt0KVbZIYH709PR0f/0D9O57978+Cy6+j4Iwyj/7p1xduW762+0ePqg/0/1et9/rf6YuPvsdrnVRBjmm/+xf8+o/VqsyXkWD3qMnD548fHJ0+NDrdx/g2+PWZ3fX//urSONlEc/m5X3fj5O49H0vu/wnnP+jBw9uPv/43HvYf/Soh1b9Q5z/w0eHh5+p7u95/vM0LW9r97Hn/0cv7w+h/w939X/vTv//Lvr/ka3/j/rd/hPvUb/75OjJnfr/19L/0ygo13lU/N76v999dGT0/6Ne7+ERzv+D7mH/Tv//Hle73f4uTz9EycE4mCzGaRKpIAnVHP9N8mBaRqEygqGmaa6iTZRfqkkwmeNJNk/L1Gu1zqJcPqvzSBVlmkcdaYxmyox73FLqL+PxV94kTzOFaxUFiYpW4ygM42Sm0qkq55Hq9dV0vVwe0ITUslD6cqYxiCujCyLGNYOlm3N+ujMOCbZKQe0mjs7V9uVM0iAviNh8PZEBadlE9E/xm4rmQgXLIjVzZWW6qAjPghKLw50oMVPeNJszjaKw4CZz8HIVZPQ3dFvZcl00WD1Jl+k6v68XCeqCMi7KeFKAYOIaz0yEmqlccP8NrTZYRWrKG1nvV5aeY2No2iyPwnhSxmnCMxcdYVG8ipdBfoDO2TJScRJGFzw6HrY0pR0wkhs/O/uriguVJstLlWA9oDZNJnqjyzkegeg8kEmyoCi8FmSrBZpWyvenayLJ91W8ytK8xCxJSotLk6LV0vcmGG+d51FSetIc3IcMTM3zODWffinSxHzG6Z3Gy0gmAv+WEZNQmJnCaBqslyUtX9pg4+bLeGyen+FrTcKmbz4m61V2SQQkWav1w48nz05fnL7xfzw9eaEGuOcFeR5cOsOu9+Dxw47Cn4dH/Kd7NOrQ8+kyDcrDvlv3ff3m262u/f4T6tPvP5A/D7e6tp6efPMfT1++OH2NjlctkqV2NJ0mUemPu+1j5bRhulftDt+NJzFYJ8+8PACa7y3abkd64WSkm0MfLN4k2Ce/5N7zKfWdBpNonKaL+9LowDQ6KOPk8gCig22Nk4PlJuwdPX6y2h5zE5dF7+iW8aTB/oGuWy9/+tl//vLFM6wQsO+BUp/X5whCBSmIP0DWylSEbJniiBdxGClnhW2NSXBx/HpHrhpHEMWIhdWc31ar9bk6+Acv+4S2IE72dyefjY9pz5KQd9VVB19ZX4+FU+12v7d9tiFf4ygvjtXzYMzn+n5Rhh3FizooJgHW9Tx9pqIkymeXHTXLg5D2t6CP0aUCL8oUN1c4oPMgTM9x+uXweXTsaNolBh6QSHuTTfkNZs+J2g7f+ebl85ev/FfPnvafnzx1PXB5HmSRc9DrqEPXC4ryEt9sUaQBZ79iuGevTt7tG0DdV/2HD70uD7QJcgw18/DXcdWXqhcdHAnJ6cyP8Gg44q+kWnCQE+WArn5H4Zg8doWndJnlPQvWRREHydPlOndmHeV0cZxcMMatmvLAXpBlURI6TJPjUNfnQbYMJujrmJX85B/2v3PVvXuq73q0L6DwHqjAjQdYBEh2NTcusBWXmoTX6Tha0tz1GB3VEzJufA4yezLWKpiJaije56Uzu8CM+O9LGh+fLqURbbm0qjafRhzHSTF43FF5kMyigcODusPuCMTOPDo83BnyuEBnvfSZ+gsUzhEGx2gr2AYsc+aa9cp0OK7rPKEGpJiDMkrwzxlCqqQZLY2+QGzlM7G4o4ZYix6nQ+vi5y4UGxGNx0TIaLRXxj7JYa1Md2uyhBVSTy34wUfxr0kMuVqp85zEIT8W6OAsibvQJBBitY6T8rHiE1zwiXayNF1CCw1fdL7FUnYAAN3PRgri+gJTuXwEhe1QF76fROe+70yWOLwJDPUxwY6Ouhfg372FJdDxlJ+rwUC1i3I9btePrB15QZbpNR47tXzrR8UaK3Jcz5rStQmRCI9TRMtpg5Qw2sQT+QIhaU/WYQBNPs16R8cKqnyJm2/ydWSTqq1vmk/m1U0a1+NbHflMU+iPMgUG0s/lkdxtDkDTkqjSH0Ij0sYjnF4W53E5d4TAevULQJeOglFI6IBURtObRaUj8zjCz47sj9uckLqjH/2xt0Jugx1sZJtbYZaPJ437PN4qDSNiGT0F2gWQiny+5zCJHWVMYRQOiK0dMgY+i2tUDLo1ddHSJgPWtUkEwxkMlBQk0DAmhqyTdZn+SBPeTFvVxKNR/JoiodHd7Zn4eURaCmfDAWODssydekhSEtMY6qhNa0FLHKco9/l4FG1ShHQ6unvGjQuCECR2+NMmbc8UWEwooq1zEMSA7j8Fy3V0muewQ9P2Olkk6XlSnX51Rft+3d7aaSE12gRLHJIydUS07CMCNVekuT4h8WpWWDJ/oXV0iUmcoROvbrFz6kA1QCM9aCBBsm3xipeLWUa7R7k6S7JDjEedC9fDCV+toYmh6A/ZxvR5KdYhs9Yjuq1ezbEiPTesEcrIWh/PhjXWUzeWzrcNg5g11WM6ldLdi5NpBAw/EZEnIyD3AwjcBPxyjHYJiXUDvULiXg8AGkZmDEU7qBSBu3XupvaR3X82t5koRw+cE+PnepNsjf+Foa6ohNsn2Tl5dKXr0jBFZsjii2jpb0gsi8GFu9NB25AB9fT4S+7jY4ZveynbGcDQJSdmlyKepNQTQJuU/jwOcZZ88iOj4TGkBZiiPszHo2piNti9jxFgcVYWA1VWtj7SgtlbCSRbTa0WtFxuQ+h6YRScQCxAOh2IqZ3lsfjpdMpjAAx4AusCfB1fNjzsAA6jEUXXU6+YKPjRMD3nHfWtW8Hk3yr5Q9A7+iOIPqSS7eOtMnG7jO6Rku4tUnK7dBBnMRvY47E3wRj0vqK11fd6+t5NUlNWvohs1EHPvV33ty0cVnBkQgWNGE6bgKVgwQo5VWDwDe3BwTSPONiShAfQxyxc0+lSwk1FCaU5BdtCQtghBRDy9BcJMojneQGcIkSIi2eFbSoh0wiFN6mjQZ4ymEQwgHy7Aa1ZssA787OYIqHI0zEOP09mgOJeAjQAC9dl6+CA/f0H7n7AXZu9rQNpzff3bWsmXYbHczgZ1Pb4nD+M9tBhbzk1PZc/h02nkdcOzhay9IZTkokCc9h9xbnL2OvQX0dNKar9Gmz43JEhv9ZsvcV0NgzlgqgYar0gvItX7rZRb+2o/ho+lA2nu9R8POjBNVoFF9hGCazRWDTdLkAw+vXXDSke286YH9HL1ppti2raUQscLuovcRrfhG624yIdDtz4FLg5JgQJVpjAz96QyY7QHfd1SAB3q6EgH8QraizsScixoI50m2SIwGqeruH0kxxOFcmo67Iksnu63ep8p5XtC5PzLguV0IeTQFSTucsjRHmWLjmkOaB2P7x4c/rKP3l1evKJHFvwIYcOQvgWUW7meUwIm+PuhQPnyKfIpkF4FNY02A5K6F0cLUOEzIDbO1V8zedFDCmy7LNRU1kVyGffgoNwPG9Brtok5QA7pCeAqSVKWAgrJYcWtDvBahwGCtvfiBXRl3jFg8jhxwTj9RQWk0IvuMFOtw6X/PAj+Patz2El1w4xPX32qg8PXXaFTe9k6r1hWs5AyulFNIGxzZ0HLoVuo4taflnyMxb9ilUNyzKDBGRgH0WKh+IV49MIK7JCyA7917RIAgAkCu2lCDA5Zcaz77HGRASrhxvQ2yLCSfE7FEonIWb/2cvg98YkV07ba+8HZUL7EN1HQ/QloktPx+OJLmdFSjYIt6w2kcNT8qbKIB6kalU47i6BJDwYmILvHjR3WDizYZu+tUe7VHHqxqcI3XA2XIxkKpoFEIripjOXXeymV48+bXe0i7Q3OGb3JBk0YPF2ogtoyQz+26RDE7SNSHu/ZLP2yLT2IdO7pF3yWTBHAUNzW63HNNP2n6dOHV3StyEno5vCJwiIDbIcUt1pkLBE1qUUJfifDI7RhaMSpAdJvKpD+2qd6Hxb5dZ+uT8zR8tHqmoph7eo4S6NRyhFYlno/gVneb5QeXpeoxPGhsTbK5xaEzCjY6n9Sd69Me1etf5r0cYgoHFkiCnbR4buyS5IhqkR3k06tVYyW+HSTBFgZpSTdd+r5txm2IyZymg4UV8NNI8bbB9D/hetWqTIumzZLFDgNvQFOMCnVfiz/2RkE3KCJnxcBTLIGpqNUmqUWo2G6Wbr1DAvh9P21fiaU6TtkYlXZxNjv2/uAdqtDilAmrvtPRAByKAg8SZiR9wygRykEdqCTiG43yHpGe1RAPZ8lAu1JiybM0rLNsnq1loSDdWGdhJlIhI2IV4z90YUMNo/XGOh9iDpxurBkmZaXS2O+bg39JDThuajdJVkP/045C/kA9EHaAF5mi2ZJ+1pugzb7rVrS52TUPLCVf+m+t0u+WTdJtOgARCbUghGUbtrfTq1Dpf4AdFWIbjN3swJk9xRG6KaOWEE8dqMM2zzetsjDs/hU6ueXSs0aIurJWwTP3YNKX/WiuFYtUEhLdP7JY0TdLtaXF9tBHtdtxskUHBBE8AqXP0JKk/mb2YO0FAr1SLYRL5RVs5UFEaHM7HHnIBl3feiitJrbSXc2TQZcNPcwgxiJSb7gEznCjFNBFQhnJigI3zxyVwNqhQsm7IQTiGIqnhI/sO9ezr4rxdA5q5ewBbdtc7+IAifWutZoZPTcz+LJ4tlNJBztbX5HxpS+cEjc11sLY7Jbu/dbssaQwE7H4Z2j9Huhvwx6//u6r//GPXfj7sPDx97/QeoxHry6K4A8F+q/o8Lgz598d/H6/9Qbnok9X/9Q/zfp/rvw+5d/d/vVf93tlUchsj6dikZPFKKGXN0o0OFWlMOMUrYA0/QgBKIMbWS+kGrzo1K1IIcSBMwA0ULG+Qf7wMSoNV9RTV9RVW6gvB8sV5RrQ9SEep7JiZArQxC/gjzIxZNFp3b6WQlSG0xkjlYRjQueTXK4bJDU49Iso2GvfuJ9iHIHUJFkhLcx/Vt4mTpMrgJAkF4HL1fw3hfSl2gXj3FlkvTTmZ0uBaACwCrR18Ueva67K4AqOBAwcMDApLqm5/E3UZF0biaGXynDAacLsWw01OnRBjqnhB1mSO1QPD9m7O/HnARHhfqBQuwr0BQJQl/e8Edl9Fx+zAA/pT8c1U3Z2619lXE6VsZiAi4TC8LW63vXp188+aHly+oWm3Ylq0WhJ0wuObNbo9aJ8/Pvj95bbDSrMhQMMYhS9T39HqfKFxG9Cta0Wq8vGQQx3vil5RlcSzHtaPZj69Z6H2Lbt+hskbccftG5Za/hO8EQbOCZbyztUxh/1EYQx61U0s2wF3X8+AscMqGtvc+SULlg8checb2hI52ewwu5WaEijW9XoFaP37oWE6MW7nWU5qz2hPLXQ4vhlN2FcILr+rI8RR4dyOKuve6XR11p8ZtcknaN/XQT926+Tm37SEJfp97sKSPLxtUevC+vKp2AY/gibcbaBU9NfrWWsjHegt75yr0LbEHK6ld7dWZpRxCuPszOPMBFNUEgo6NIy8BCsZjr/PLqu7Qyi2EvM0m7FHt1njMGyGxrCuEsWhDOT5H6ZEKzjOxBObxhL4vePcbbgsKBa7Ys21r35JWSb5B23Ju2+wkznWcf9e77uy6yCPtHtahm3FhJ5HQejzS5WSNce0IRGc3xqA3mmo76ZiPGxPQUseNmJ7UgJqoHkVq4ITSTVd9pXpbBJmS0nFazttbxBmKq+mo7ci9ZY2YzIpVVNtxvFWCYpbncwqE1DlPvRvsUFSbul3E/QWdbKTwW40cSVl8GiUmho/maX1dK2RJZr6iZ2QjZUFEuJ1xEVmiOtKdu5M0mto3hROfKyfkpBsFaSkXirA/l96yftbxO5j7SZSVjSH50dec9JwgzDJPwyqrNMWheNtR7zinFyyzeWCFlVZrXQ6NY4c0PjUMLmIUQHW02S4GVjyuCK0KSauX8xYnaLWWis19IzQKTOn6GwaqegGJ1EVfl6smRe9up+jcoujcpfoKCmHU8euTjnqKJn+j+lGs33mHOS9XXE56XjWiZf2tyqLawhuqv6BOq0q6NuWW9lBb0DgJliizTJebyDnx3iCXeYIlM7elwDO6jJwQAEQePnWbpVW08yGwjpKqyHnEKARgg5DXiiqqqzAzME6iI0f7iJHxd0gigvBkh6R6aaDt6U5+s5JvFHrDRocdnqZDHKzTlnlwrrOVb3dzlSIbUv2A08GbTV/4VLgm58vE64KHSsCtxKhguJtnwVsUMGoXUoBF9Lx1LQLXCc8WbmfxYUh+FpES5IhQJTBdGOeoLMAH2CYMVXH+WFGh8VuQ/DNIHeuKFwYgXHgg72BUmqlZ3PKzKQHhldo82BVZvaafO1v8sPlIVNQLBKRKsg/by9PjXLVZnx7XnQkO0tTmHn/BTaLN3JN9bleTm/vVjWsTV9SsP9Mzn+nSgGWcOWdcX82JlQo4nXlwL6gqYBFFWRivCi6/3C53PifpdwrYJwxwBoZRzBp6YIW+cHSgS55AYAXenFUn14QKsSm+4BwnnDYhJUbbU+iErToRfbPrNfAWy2h/5leRqjIZen0n4xRsAFNdaI8DiDOM6VUQ5F7gVkxwemtZKLfx5RkdqeV6hfrxCiYKMxBirLEa53OmFu6T8qHqlYByL8CzmTobVuOPqtrwxk0uEqfY5zIJHGBPt4oc84B4SwtgIjQSpocFUftmrsfFWTHwFEVMcY7EoRl4FUR+ljnE6+1N4sn2OAN1nJdSdtzVWgJOCHeERzMZ0jOPAV6nRuBU9D4u+HUC4Gs7DDxtEzlX02tIutRaRQDiphp/D5i3QsHUswYt0lnvUKOcH2CW9moviQbgEyylZDO9SxQs27uT+AnpIzyyJhvinp5wtG/iRjbgE2AiFvcDTBSH7M/ydk42fppOCUKEU4006M20ZVgM+L0I56FbH7eX6/IgnYojvnvk4IWTB25KDHksq8SBEocOHQdC/FyHh3vwZ6xDodUQvcboOIx3py4hK3FRYIi41Ts+VvYhsIaoPQhag1V5lHNydaP+nYngJWDXFo35uWG023CwryGVPVam1mPANuSk9jv5g87nuo/cEBhX9T8blhFJw9wzlvIt3Wns/Zk+dNrdcoqodr53lWSVN+dTV65xrofNFlyYUW0nYikSwNHWkvC3ZMRJSwrooKrROiseUOoXKhK2l6IB5p1HvEGGd0CkAyveqPxtG8+nq3LQK601REPDEASoOjIPpcE76kqO8udwMWHjECA5BgHnlP3S7KxCQmx6HaKLNtP9DZJWCZM5Cn+HPJ0NbxGmURM275NqLTtbaHpLuIRcWjiqYMha2baUudgRQixuVknetgFLUKJtm3nCOMNflyJRvMdoR/CAsn2WNdD632R6uUaBhe4tO0ZRuacOgLBxVdVgegXUXkJdTeh+xu+MNfTVVrXAr2aARYDhg7NnLTRD3QG5fqrjWSHPy6FdRGIGumgLb0PlwOO2Van7kdQOiRe18rcIv3EvhHsVy01HZvt14w09nSunWFv/cXEtG/fjyam6Wg2/MBR9MTr2+lMksDPl8JiDq+DYO5xduxpzlPzaYwPoEG2uR9EaXfrs1Avk6tMqkBYi7LELCjGkHFqtxxZJ4httd+q/76hTFEpjH9/Jn8XgYW1sXstL1nF5eVAFo/VBMe+NL1C2DLUInYDuUkFuYnz0onoRJxaKo7gp5gQI50m9N2LdE/Nq8YyW6Ry81oetx37p8UJkU3usFDb2Ay6jpEYOGicJvTco0Jm85cb6HV7YMElGwC7nQ8/ztN/gCqAmz+p8P7amkZtYrnpTvxDxr1n1vH8gpcsxeTfykh2b5/rl/hoPI7aKQ1ZlHW6BuW9/I6Y9vRHTalwj77DSPtjd6MZGvAXt/9JinI3ZiG3OVLx+0uB1g/SNrlsYnApUcy0BnGyM+iB52y0eM5w5CIqDGt90rFyGVC5Wvx5ASAMJmDnlUOhuQuI6RkG7JGFo7dgC2Bp3yyqSatq2i6NbDGK0cStG7xGK4c1WVmrtjGYmK0Tf9+8VR9c16GfEq3k4qoxM0SxFm+43kTjeZCRPh8bsTUGF/vYn+lbbSFHatnZ4rwnHGNWnGoPnuz5C1WpRq0amtSq12vLgRD7em5F2HbqGdBnjEOr6eTYTlJwSQ/GJEPpuflDDP33fJ5gpBkqcWD0/kS+qcdD18NMIUVYM2IOvpPpFrR9EQqoxj6vJ6PxjbPU//63eQ1852k/+koajdwMIQZl35SWOoVtAGsvQKGU7rWi7B3Ro4sQcm4zCq0ZXn8/TIrLjY2FcBDN6twTJT3REgU3NEY/IYZVQ1DcpLRpJDK6u5lQ8hmdYIB4ol8Jy36n8sQr7KKwHp9LY2voVb1kn53g4Jnp4s29J9oQAJS1dJGi5WWrUgNAHA8xJFAN26hq4e9XekTXQyPM9ebfXH81I0Vqc/a7plBxrdrWnlf98X9UrkW2lh/WBlexWtXgkcpOSiqBxqrAIen2bgtSD9hz6Lcq3qtau2mYdSBVS1hqYxdwBjHmPr+/pBzcywjb4n9549RHEIqRzXfk5RtA5rLcl6pOMsuVcgJ9lgx5FO/RLD8Fyum28aBsn2RDzjmhZ95pLpydEyYj3lOMK+3bULJCTW2A1T/RVTQQYm1w6Gi/YzKCGhtidteUR5YBvOMe7a2xaKHtkwsO/nmNCYJTnnyoIw7vLb+c4NCqC7kQc2sGpsnkSTKjK9780N2yRqaO6FOukt0mqo+PWxd9tKq0sfSNMFRbW85ugidWB4L9RVXa/+mwTxSQKmmg+JLeEjayxRSX41MuHairnYG1jaD2e0R0fHZDeEwGqxOsXPjiFKC2nLTXcn0z2LK+KKu20p1VRn/1hJdJ59JSE1hVtx47cnpGny2A22xpYNvCWsXWDvcObI45VUvmEk7JcI//ciExahaWpmOikHPQ7pix+wMR8Imu7hruQlyiHQcR306ewSh0ARr6bjO/nyAhZyLq2z4x2yHY0DKY1ghhNT/1Qyi8/zaOl2ByOSmzPhIkgrFw3QxU6k4UAyYhe1ORCm4ICNVQilEcmGEQ1PBQcrCNylrFtBGZiMpdZ6bVef/Py1akUmWAplKvkCm2p7xCdQd8JhzW/GXtsfS+BVGtMzS8DOqfk0YlDp/25Xvd2h676XS6NJD7u2H2hS0SAR+hHwIiSAI+bbh57ebWTt9D2dyHk1emz3+ICFh9xASs/sbCcQcvlk9glzfVO+4StKpnr4D4cxF2vUKdg7BSuQ40pB1sloSSFe8MATRSrE4I9/p2Fwry9rjeyFktHLH1HR1RsARlI6AebvoUwnYST4JZs24BRR3Yos0LuqIxfbdouaMJJkTbsylTSerznbVR6q8aR2ff1bUj3vv5BIUXtdkNWZ/aNXZy4b2bD31B+cQhM2kOQdbx2c6LhDR2qE7h3BTDM9kKIi5Kmv3m0/bwENEEAyqHnlXjJz2VoOdp+i1wGNhJEutJveivQYeBH00GphUae4ifQRHiwELhhpdZ8jTKwugaslrA6MP2efZ6mi9KE/DWYD0041YD5hghi5n8MttvAmWARQHftQzGMNsHcbTjdRNMEmNkV/gBlUgEHevc2kx/i6qj3Lr1zoSH1R9rS/NyaPbPKroMg99rePcY18HQZPdYb9T1uHzDc4Yq8jPLJScm5VwJnRUdvYcX1942f7LOQ+C4YGrl/rjfJ6mAA+t4uNrPE+Goma4CqQ2cQZ4hi4iCrZV7KzvE516HO1xFiPQWiqvSLAAvHraII9bPx1jN74spVglrLKfvuyOAu/xpEldpGAJ1+92AT6SJKiAVY3FGa1/xLdLB2+A0yr8+/icj/PXrIitrKv50w3JPYfFqIDy8/3sQDkPWswp8JGImfsUQrpOB5Jtk+91gOF7/cqOGj2vDvBdZJGzJjmjh+qV6f/jQnT7xhLHVZDyXNZFWVTTGx7a1fpZtzNoDXWyughXnz3H6lvKA3z/lIQnBrLxVEk1PzC/1cC9EzhIHu6I+L49ENcXVZqs8TU1kEXC2A3YsdGC87ej7U40GL9lC519n78rDA5V0MTgi8SkJ3pNUOngaa/mVkg2daKe65lXVBGur65ggnh+bv3jS5u+6uu+vuurvurrvr7rq77q676+66u+6uu+vuurvurrvr7vrk1/8CRWlXNwB4AAA="
WORK = "/kaggle/working/soilsight_src" if os.path.isdir("/kaggle/working") else os.path.abspath("soilsight_src")
os.makedirs(WORK, exist_ok=True)
_t = tarfile.open(fileobj=io.BytesIO(base64.b64decode(PAYLOAD)), mode="r:gz")
try:
    _t.extractall(WORK, filter="data")
except TypeError:
    _t.extractall(WORK)
sys.path.insert(0, WORK)
import numpy as np, pandas as pd
from pathlib import Path
from soilsight import features as F, heads as H
pd.set_option("display.width", 200)


In [ ]:
# ---- load model config, features, labels ----
from huggingface_hub import HfApi, hf_hub_download
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    TOKEN = os.environ.get("HF_TOKEN")
assert TOKEN and TOKEN.startswith("hf_"), "HF_TOKEN secret not attached"
api = HfApi(token=TOKEN); user = api.whoami()["name"]
CACHE, MODELS = f"{user}/{CACHE_REPO_NAME}", f"{user}/{MODEL_REPO_NAME}"
cfg = json.loads(Path(hf_hub_download(MODELS, "config.json", token=TOKEN)).read_text())
feat_file = sorted(f for f in api.list_repo_files(CACHE, repo_type="dataset") if f.startswith("features/"))[-1]
feats = F.load_features(Path(hf_hub_download(CACHE, feat_file, repo_type="dataset", token=TOKEN)))
samples = pd.read_csv(hf_hub_download(CACHE, "data/split_samples_seed0.csv", repo_type="dataset", token=TOKEN), dtype={"sample_id": str})
df = H.photo_table(feats, samples); sets = H.feature_sets(feats)
FR = H.FRACTIONS
X, a, R = sets[cfg["features"]], cfg["alpha"], sets[cfg["retrieval_features"]]
dev, cal = (df.split == "dev").values, (df.split == "calib").values
print("model:", cfg["features"], "| retrieval features:", cfg["retrieval_features"])


In [ ]:
# ---- out-of-fold predictions and every difficulty signal, at sample level (dev) ----
oof = H.cv_oof(X, df, a)
p_dev, s_dev = H.per_sample(df[dev], oof[dev])
ids = p_dev.index
y_dev = H.sample_truth(df[dev]).loc[ids, FR].values
E_dev = H.sample_embeddings(R[dev], df[dev]).loc[ids].values
fold = df[dev].groupby("sample_id").fold.first().loc[ids].values
mu, sd, dist = np.zeros_like(y_dev), np.zeros_like(y_dev), np.zeros(len(ids))
for f in range(5):
    q = fold == f
    mu[q], sd[q], dist[q] = H.knn_stats(E_dev[q], E_dev[~q], y_dev[~q], K)
err_dev = np.abs(p_dev.values - y_dev)

# calibration split: model trained on dev, neighbours from the dev index (as in the app)
head_dev = H.RidgeHead.fit(X[dev], df[FR].values[dev], df.w.values[dev], a)
p_cal, s_cal = H.per_sample(df[cal], head_dev.predict(X[cal]))
cids = p_cal.index
y_cal = H.sample_truth(df[cal]).loc[cids, FR].values
E_cal = H.sample_embeddings(R[cal], df[cal]).loc[cids].values
knn_cal = H.knn_stats(E_cal, E_dev, y_dev, K)
err_cal = np.abs(p_cal.values - y_cal)

rows = []
for m in H.SCORES:
    d_dev = H.difficulty(m, p_dev.values, s_dev.values, (mu, sd, dist))
    d_cal = H.difficulty(m, p_cal.values, s_cal.values, knn_cal)
    rho = np.nan if m == "constant" else np.mean([H.spearman(d_dev[:, j], err_dev[:, j]) for j in range(3)])
    cp = H.norm_conformal_fit(err_dev, d_dev, COVERAGE)
    h_dev, h_cal = H.norm_half_pp(d_dev, cp), H.norm_half_pp(d_cal, cp)
    thr = float(np.quantile(h_dev.max(1), TARGET_ACCEPT))
    acc = h_cal.max(1) <= thr
    e = err_cal.mean(1) * 100
    rows.append({"signal": m, "rank_corr_with_error_dev": rho,
                 "calib_coverage_joint": float((err_cal * 100 <= h_cal).all(1).mean()),
                 "calib_median_halfwidth_pp": float(np.median(h_cal)),
                 "calib_share_accepted": float(acc.mean()),
                 "calib_mae_accepted": float(e[acc].mean()) if acc.any() else np.nan,
                 "calib_mae_flagged": float(e[~acc].mean()) if (~acc).any() else np.nan})
report = pd.DataFrame(rows)
print(report.round(3).to_string(index=False))


In [ ]:
# ---- pick the signal that tracks error best on dev (calib is only for reporting) ----
cand = report[report.signal != "constant"].sort_values("rank_corr_with_error_dev", ascending=False)
best = cand.iloc[0]
METHOD = best.signal if best.rank_corr_with_error_dev > 0.1 else "constant"
print("chosen signal:", METHOD)

d_dev = H.difficulty(METHOD, p_dev.values, s_dev.values, (mu, sd, dist))
d_cal = H.difficulty(METHOD, p_cal.values, s_cal.values, knn_cal)
cp_dev = H.norm_conformal_fit(err_dev, d_dev, COVERAGE)
REVIEW_PP = float(np.quantile(H.norm_half_pp(d_dev, cp_dev).max(1), TARGET_ACCEPT))
h_cal = H.norm_half_pp(d_cal, cp_dev)
print(f"review threshold: flag if any half-width > {REVIEW_PP:.1f} pp")
print("\ncalibration split, accepting the most confident samples:")
print(H.selective_table(err_cal * 100, h_cal).round(2).to_string(index=False))

# final calibration on dev (out-of-fold) + calib, for the app
cp = H.norm_conformal_fit(np.vstack([err_dev, err_cal]), np.vstack([d_dev, d_cal]), COVERAGE)

# single-photo range: same signal, calibrated on out-of-fold photo predictions (photo_spread is undefined for 1 photo)
m1 = METHOD if METHOD != "photo_spread" else "constant"
Pdev = oof[dev]; ydp = df[FR].values[dev]
Rp = R[dev] / np.linalg.norm(R[dev], axis=1, keepdims=True).clip(1e-9)
pf = df.fold.values[dev]; s_of = dict(zip(ids, fold))
mu1, sd1, di1 = np.zeros_like(Pdev), np.zeros_like(Pdev), np.zeros(len(Pdev))
for f in range(5):
    q = pf == f
    mu1[q], sd1[q], di1[q] = H.knn_stats(Rp[q], E_dev[fold != f], y_dev[fold != f], K)
cp1 = H.norm_conformal_fit(np.abs(Pdev - ydp), H.difficulty(m1, Pdev, None, (mu1, sd1, di1)), COVERAGE)
print("\nsingle-photo signal:", m1, "| q:", {k: round(v, 2) for k, v in cp1["q"].items()})


In [ ]:
# ---- save to the model repo (config.json only) ----
cfg["conformal"] = {**cp, "method": METHOD, "k": K, "coverage_target": COVERAGE,
                    "n_cal": int(len(err_dev) + len(err_cal))}
cfg["photo_conformal"] = {**cp1, "method": m1, "k": K, "coverage_target": COVERAGE}
cfg["review_pp"] = REVIEW_PP
cfg["target_accept"] = TARGET_ACCEPT
cfg["uncertainty_report"] = report.round(4).to_dict(orient="records")
sel = H.selective_table(err_cal * 100, h_cal)
cfg["calib_report"] = {**cfg.get("calib_report", {}), "conformal_v2": {
    "method": METHOD, "n": int(len(err_cal)),
    "joint_coverage": float((err_cal * 100 <= h_cal).all(1).mean()),
    "share_auto_accepted": float((h_cal.max(1) <= REVIEW_PP).mean()),
    "mae_mean_accepted": float((err_cal.mean(1) * 100)[h_cal.max(1) <= REVIEW_PP].mean()) if (h_cal.max(1) <= REVIEW_PP).any() else None,
    "mae_mean_flagged": float((err_cal.mean(1) * 100)[h_cal.max(1) > REVIEW_PP].mean()) if (h_cal.max(1) > REVIEW_PP).any() else None,
    "selective": sel.round(3).to_dict(orient="records")}}
api.upload_file(path_or_fileobj=H.to_json(cfg).encode(), path_in_repo="config.json", repo_id=MODELS,
                repo_type="model", commit_message=f"Uncertainty v2: {METHOD}")
print("config.json updated in", MODELS)
print(H.to_json(cfg["calib_report"]["conformal_v2"]))
